# BindCraft on 5G51 — free Colab, crash-proof

De novo binder design against **PDB 5G51 chain A** (DWV P-domain, author residues 260–416),
**free binding — no hotspot constraint**. AF2 chooses the site itself.

---

## Why this survives being killed

Free Colab *will* disconnect (12 h hard cap, ~90 min idle). You cannot prevent that. This notebook is
built so it doesn't matter — verified against BindCraft's source:

* **All output goes to Google Drive.** `design_path` is the entire persistence mechanism.
* **BindCraft resumes natively.** `create_dataframe()` writes a CSV only
  `if not os.path.exists(csv_file)`, so existing results are never overwritten;
  `check_accepted_designs()` counts `.pdb` files in `Accepted/` and `check_n_trajectories()` counts
  `Trajectory/Relaxed/` — both read **from disk**, not from memory. The maintainer confirms it
  (issue #40): *"you can just start it again and as long as it points to the same output folder it
  will continue doing the designs."*
* **Every trajectory uses a fresh random seed**, so restarts explore new space instead of repeating.
* **A supervisor loop restarts BindCraft** on crash/OOM.

**Resuming after a disconnect: run cells 1 → 2 → 3 → 6.** Skip cells 4–5 (settings already on Drive).
You lose only the single in-flight trajectory — maintainer: *"you will lose one trajectory by
restarting but it's no big deal."*

---

## Read this before you start

**Throughput — set expectations low.** A T4 user on a 379-residue system reported *"almost 3 hours"*
per trajectory (issue #239). Easy targets need ~100 trajectories, difficult ones 1,000–10,000. At
~1–3 h each on free Colab you should expect **single-digit trajectories per session**. Reaching 100
accepted designs is not realistic on free tier; `N_FINAL` defaults to 20 and the campaign accumulates
across sessions. That is exactly what the resume design is for.

**VRAM is the binding constraint.** The maintainer's own figure for a 16 GB card (issue #36):
*"with 16 Gb you might only be able to design for 350 amino acids, that's binder+target combined"*.
With a 157-residue target that leaves ~190 for the binder, so `LENGTHS` defaults to `[60, 130]`
(217–287 total) — inside the envelope with margin. Up to `[60, 150]` (307) is still defensible. It is
a rule of thumb, not a measured limit, so the supervisor loop catches OOM regardless. There are no
memory flags; the only levers are a smaller target or a shorter binder.

**Do not run two sessions into the same `design_path`.** `insert_data()` appends with
`mode='a'` and no locking, so concurrent writers interleave and corrupt the CSVs. Sequential
restarts are safe; parallel ones are not. Use a different `RUN_NAME` for a second session.

**Disk.** AF2 weights are 5.59 GB and the tarball is not auto-deleted (~11.2 GB peak). Cell 2 removes
the tar after extraction.


## 1 · Mount Drive and configure

In [ ]:
import os, sys, json, time, glob
from google.colab import drive
drive.mount('/content/drive')

# ----------------------------- CONFIG -----------------------------
# Pick ONE group per run. Each gets its own Drive folder - never point two runs at
# the same design_path, BindCraft appends to its CSVs with no locking and they corrupt.
GROUPS = {
    "A_ridge": "325,326,384,386",   # Q325 P326 V384 N386 - best anchors, zero occlusion
    "B_triad": "277,294",           # H277 D294 - opposite face, comparison arm
    "C_face":  "289,384",           # W289 V384 - top 2 anchors on the ISOLATED domain
    "free":    "",                  # no constraint, AF2 picks the site
}
GROUP = "B_triad"                 # <-- change this per campaign

DRIVE_ROOT = "/content/drive/MyDrive/BindCraft"   # NOTE: MyDrive, no space
LENGTHS    = [60, 130]            # 157 target + 130 = 287, inside the ~350 T4 ceiling
N_FINAL    = 20                   # accepted-design goal; accumulates across sessions
FAST       = False                # True -> num_seqs 20->8. DEFAULT OFF: num_seqs is
                                  # shots-on-goal per trajectory, not a speed knob.
FILTERS    = "default_filters.json"
# ------------------------------------------------------------------

HOTSPOTS = GROUPS[GROUP]
# folder name encodes the hotspots, so the campaign is self-documenting on Drive
RUN_NAME = f"5G51_{GROUP}" + (f"_hs{HOTSPOTS.replace(',', '-')}" if HOTSPOTS else "_freebind")

DESIGN_PATH  = f"{DRIVE_ROOT}/{RUN_NAME}/"     # MUST end with "/"
SETTINGS_DIR = f"{DRIVE_ROOT}/{RUN_NAME}_settings"
LOG_DIR      = f"{DRIVE_ROOT}/{RUN_NAME}_logs"
for d in (DESIGN_PATH, SETTINGS_DIR, LOG_DIR):
    os.makedirs(d, exist_ok=True)

os.system("nvidia-smi --query-gpu=name,memory.total --format=csv,noheader")
print(f"\nGROUP     : {GROUP}")
print(f"hotspots  : {HOTSPOTS if HOTSPOTS else '(free binding)'}   <- PDB author numbering")
print(f"folder    : {DESIGN_PATH}")

acc  = glob.glob(DESIGN_PATH + "Accepted/*.pdb")
traj = glob.glob(DESIGN_PATH + "Trajectory/Relaxed/*.pdb")
print(f"\nEXISTING PROGRESS: {len(acc)} accepted / {len(traj)} trajectories")
if acc or traj:
    print(f"-> RESUMING. BindCraft will continue toward {N_FINAL} accepted designs.")
else:
    print("-> fresh campaign")

## 2 · Install BindCraft

~10 min on first run; afterwards it detects the existing install and skips. Downloads 5.59 GB of
AlphaFold2 weights.

This mirrors the official BindCraft Colab install. The JAX check at the end is a cheap guard: if
ColabDesign's unpinned `jax` dependency ever replaces Colab's CUDA build with a CPU-only one,
BindCraft dies with *"No GPU device found, terminating"*. Better to fail here than an hour into a run.

In [ ]:
import os, time
os.chdir("/content")

if not os.path.isfile("/content/bindcraft/params/done.txt"):
    print("installing BindCraft ...")
    os.makedirs("/content/bindcraft/", exist_ok=True)
    os.system("git clone -q https://github.com/martinpacesa/BindCraft /content/bindcraft/")
    os.system("chmod +x /content/bindcraft/functions/dssp")
    os.system("chmod +x /content/bindcraft/functions/DAlphaBall.gcc")

    # AF2 weights in the background
    os.system("(mkdir -p /content/bindcraft/params; apt-get install aria2 -qq; "
              "aria2c -q -x 16 -d /content "
              "https://storage.googleapis.com/alphafold/alphafold_params_2022-12-06.tar; "
              "tar -xf /content/alphafold_params_2022-12-06.tar -C /content/bindcraft/params; "
              "rm -f /content/alphafold_params_2022-12-06.tar; "
              "touch /content/bindcraft/params/done.txt )&")

    os.system("pip install -q git+https://github.com/sokrypton/ColabDesign.git")
    os.system("ln -sf /usr/local/lib/python3.*/dist-packages/colabdesign /content/colabdesign")
    os.system("pip install -q pyrosetta --find-links "
              "https://west.rosettacommons.org/pyrosetta/quarterly/release.cxx11thread.serialization")

    print("waiting for AlphaFold2 weights", end="")
    while not os.path.isfile("/content/bindcraft/params/done.txt"):
        time.sleep(15); print(".", end="", flush=True)
    print()
else:
    print("already installed")

# --- preflight: fail here, not 40 minutes into a run ---
import jax
devs = jax.devices()
print("jax devices:", devs)
assert any(d.platform == "gpu" for d in devs), (
    "No GPU visible to JAX. Fix: Runtime > Change runtime type > T4 GPU, then "
    "reinstall jax with:  pip install -q --upgrade 'jax[cuda12]'")
import pyrosetta; print("pyrosetta OK")
import colabdesign;  print("colabdesign OK")
print("\nPREFLIGHT PASSED")

## 3 · Prepare the 5G51 target

Downloads 5G51, keeps chain A residues 260–416, drops waters/heteroatoms, and keeps a single
conformer per atom. Cached on Drive.

**The altloc handling matters here.** 5G51 is a 1.45 Å structure where 21 residues have *no*
blank-altloc copy, so the obvious "keep only blank altloc" filter silently deletes them — including
291–294 and 303. Verified on the real file: blank-only yields 136 residues, `(" ", "A")` yields the
correct 157. Two asserts below catch it if this ever regresses.

In [ ]:
import os, urllib.request
TARGET_PDB = f"{SETTINGS_DIR}/5G51_chainA_260_416.pdb"

if not os.path.exists(TARGET_PDB):
    raw = "/content/5g51_raw.pdb"
    urllib.request.urlretrieve("https://files.rcsb.org/download/5G51.pdb", raw)
    out, seen = [], set()
    for l in open(raw):
        if not l.startswith("ATOM"):   continue   # drops HETATM/waters
        if l[21] != "A":               continue
        try: rn = int(l[22:26])
        except ValueError:             continue
        if not (260 <= rn <= 416):     continue
        if l[16] not in (" ", "A"):    continue   # keep one conformer
        key = (rn, l[12:16].strip())
        if key in seen:                continue
        seen.add(key)
        out.append(l[:16] + " " + l[17:])         # blank the altloc column
    out.append("TER\nEND\n")
    open(TARGET_PDB, "w").writelines(out)

res = sorted({int(l[22:26]) for l in open(TARGET_PDB) if l.startswith("ATOM")})
nat = sum(1 for l in open(TARGET_PDB) if l.startswith("ATOM"))
n_ca = sum(1 for l in open(TARGET_PDB) if l.startswith("ATOM") and l[12:16] == " CA ")
print(f"{TARGET_PDB}\n  {len(res)} residues {res[0]}-{res[-1]}, {n_ca} CA, {nat} atoms")

# Tripwire. 5G51 is 1.45 A and riddled with alternate conformers: 365 of the 1433 raw atoms in
# chain A 260-416 carry a non-blank altloc, and 21 residues have NO blank-altloc copy at all
# (263 270 271 278 291 292 293 294 303 313 318 328 350 352 353 363 375 392 400 405 411).
# Filtering to blank-only would silently drop those 21 - including 291-294 and 303, right in the
# surface of interest. Keeping (" ","A") gives the correct 157. Raw file has 180 CA for 157
# residues, so never assert 157 before dedupe.
assert len(res) == 157, f"expected 157 residues, got {len(res)} - check the altloc filter"
assert n_ca  == 157, f"expected 157 CA, got {n_ca} - duplicate conformers leaked through"
print(f"  total for AF2 = 157 + binder({LENGTHS[0]}-{LENGTHS[1]}) = "
      f"{157+LENGTHS[0]}-{157+LENGTHS[1]} residues")

### 3b · Optional speed-up

There is no "fast preset" — the maintainer states 2-stage/3-stage protocols are **not** faster
(identical iteration counts). The real lever is `num_seqs`, the number of MPNN sequences validated
with AF2 per trajectory.

**Default is now OFF.** Measured on this target: dropping to 8 cut sampling per trajectory by 60% while
only modestly improving throughput, and 5G51 fails on interface chemistry (unsatisfied H-bonds, surface
hydrophobicity) — exactly what more MPNN sampling fixes. Turn `FAST` on only if you are throughput-bound
rather than hit-rate-bound.

In [ ]:
import json, shutil, os
# BASE is never mutated. Deriving from ADVANCED compounds the prefix on every re-run -
# that bug produced settings_advanced/fast_fast_fast_fast_default_4stage_multimer.json,
# each copy re-applying num_seqs=8, so the value never actually changed.
BASE = "default_4stage_multimer.json"
SRC  = f"/content/bindcraft/settings_advanced/{BASE}"

if FAST:
    adv = json.load(open(SRC)); adv["num_seqs"] = 8
    ADVANCED = "fast_" + BASE
    json.dump(adv, open(f"/content/bindcraft/settings_advanced/{ADVANCED}", "w"), indent=4)
else:
    ADVANCED = BASE

chk = json.load(open(f"/content/bindcraft/settings_advanced/{ADVANCED}"))
assert not ADVANCED.startswith("fast_fast"), "prefix compounded - restart runtime"
print(f"preset   : {ADVANCED}")
print(f"num_seqs : {chk.get('num_seqs')}   <- CONFIRM this is what you intended")

## 4 · Write settings to Drive

*Skip when resuming.*

In [ ]:
import json
settings = {
    "design_path": DESIGN_PATH,
    "binder_name": RUN_NAME,
    "starting_pdb": TARGET_PDB,
    "chains": "A",
    "target_hotspot_residues": HOTSPOTS,   # "" -> None internally -> AF2 picks the site
    "lengths": LENGTHS,
    "number_of_final_designs": N_FINAL,
}
SETTINGS_JSON = f"{SETTINGS_DIR}/target.json"
json.dump(settings, open(SETTINGS_JSON, "w"), indent=4)
print(json.dumps(settings, indent=4))
print("\nmode:", "FREE BINDING - no hotspot constraint" if HOTSPOTS == "" else f"hotspots {HOTSPOTS}")
print("(if you ever add hotspots, use PDB author numbering, e.g. \"405,408\" - never 0-indexed)")

## 5 · Keep-alive (optional)

Reduces *idle* disconnects. It does not defeat the 12 h cap — the resume logic is the real protection.

Run this **before** cell 6, not after: the run cell uses `ipywidgets.HTML`, and importing
`IPython.display.HTML` afterwards shadows it (BindCraft issue #294).

In [ ]:
from IPython.display import Javascript, display
display(Javascript(
    'function _ka(){document.querySelector("colab-toolbar-button#connect")?.click();}'
    'setInterval(_ka, 60000);'
))
print("keep-alive active")

## 6 · RUN — supervised, auto-restarting

Runs `bindcraft.py` under a supervisor that restarts it on crash or OOM. Because BindCraft counts
finished work on disk, each restart continues the same campaign.

**Safe to re-run any time.** This is the cell you return to after every disconnect.

In [ ]:
import subprocess, os, sys, time, glob, datetime
os.chdir("/content/bindcraft")

SETTINGS_JSON = f"{SETTINGS_DIR}/target.json"
MAX_RESTARTS  = 50
log_path = f"{LOG_DIR}/run_{datetime.datetime.now():%Y%m%d_%H%M%S}.log"
print("log:", log_path)

def progress():
    return (len(glob.glob(DESIGN_PATH + "Accepted/*.pdb")),
            len(glob.glob(DESIGN_PATH + "Trajectory/Relaxed/*.pdb")))

for attempt in range(1, MAX_RESTARTS + 1):
    a, t = progress()
    if a >= N_FINAL:
        print(f"TARGET REACHED - {a}/{N_FINAL} accepted"); break
    print(f"\n===== attempt {attempt} | {a} accepted, {t} trajectories =====", flush=True)

    cmd = [sys.executable, "-u", "./bindcraft.py",
           "--settings",  SETTINGS_JSON,
           "--filters",  f"./settings_filters/{FILTERS}",
           "--advanced", f"./settings_advanced/{ADVANCED}"]

    with open(log_path, "a") as lf:
        lf.write(f"\n===== attempt {attempt} @ {datetime.datetime.now()} =====\n"); lf.flush()
        p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                             text=True, bufsize=1)
        try:
            for line in p.stdout:
                print(line, end="")
                lf.write(line); lf.flush()      # durable on Drive
        except KeyboardInterrupt:
            p.terminate(); print("\nstopped by user"); break
        rc = p.wait()

    a2, t2 = progress()
    print(f"\nexit {rc} | accepted {a}->{a2} | trajectories {t}->{t2}")
    if a2 >= N_FINAL:
        print("TARGET REACHED"); break
    if t2 == t:
        print("!! no trajectory completed this attempt - most likely OOM.\n"
              "   Lower LENGTHS[1] in cell 1 (try 110, then 90) and re-run.")
        time.sleep(30)
    time.sleep(10)

## 7 · Results

Reads from Drive, so this works in a fresh runtime even if the design session died.

In [ ]:
import pandas as pd, glob, os
acc = sorted(glob.glob(DESIGN_PATH + "Accepted/*.pdb"))
print(f"accepted designs on Drive: {len(acc)}")

csv = DESIGN_PATH + "final_design_stats.csv"
if os.path.exists(csv):
    df = pd.read_csv(csv)
    print(f"final_design_stats.csv: {len(df)} rows")
    cols = [c for c in ["Rank","Design","Length","Seed","Average_i_pTM","Average_pLDDT",
                        "InterfaceResidues","MPNN_score"] if c in df.columns]
    key = "Average_i_pTM" if "Average_i_pTM" in df.columns else df.columns[0]
    display(df.sort_values(key, ascending=False)[cols].head(25))
else:
    print("no final_design_stats.csv yet - nothing has passed all filters")

for f in ["trajectory_stats.csv", "mpnn_design_stats.csv", "failure_csv.csv"]:
    p = DESIGN_PATH + f
    if os.path.exists(p):
        print(f"{f}: {len(pd.read_csv(p))} rows")

# which filters are killing designs
fc = DESIGN_PATH + "failure_csv.csv"
if os.path.exists(fc):
    fdf = pd.read_csv(fc)
    if len(fdf):
        s = fdf.iloc[0].sort_values(ascending=False)
        print("\ntop rejection reasons:"); print(s[s > 0].head(12))

## 8 · Force a re-rank / finish early

`final_design_stats.csv` is only rebuilt when the run completes a ranking pass. To rank what you
already have without waiting for the full goal, use the maintainer's documented trick: set the goal
*below* your current accepted count and run one short pass.

In [ ]:
import json, glob, subprocess, sys, os
n_acc = len(glob.glob(DESIGN_PATH + "Accepted/*.pdb"))
print("accepted:", n_acc)
if n_acc:
    tmp = json.load(open(f"{SETTINGS_DIR}/target.json"))
    tmp["number_of_final_designs"] = max(1, n_acc - 1)   # below current -> triggers rank + stop
    p = f"{SETTINGS_DIR}/target_rank.json"; json.dump(tmp, open(p,"w"), indent=4)
    os.chdir("/content/bindcraft")
    subprocess.run([sys.executable,"-u","./bindcraft.py","--settings",p,
                    "--filters",f"./settings_filters/{FILTERS}",
                    "--advanced",f"./settings_advanced/{ADVANCED}"])
    print("\nre-ranked -> Accepted/Ranked/ and final_design_stats.csv")
else:
    print("nothing accepted yet")

## 9 · Package for download

Keep the whole `design_path` tree. The minimum that preserves a resumable, rankable campaign is
`Accepted/`, `Trajectory/Relaxed/`, `MPNN/Relaxed/` and **`mpnn_design_stats.csv`** — losing that CSV
breaks both sequence deduplication and re-ranking. `final_design_stats.csv` is regenerable from it.

In [ ]:
import shutil, os
out = f"{DRIVE_ROOT}/{RUN_NAME}_results"
shutil.make_archive(out, "zip", DESIGN_PATH)
print("archive:", out + ".zip", f"({os.path.getsize(out+'.zip')/1e6:.1f} MB)")
print("Already on your Drive. To pull it locally:")
print("  from google.colab import files; files.download('%s.zip')" % out)